In [1]:
import tvm
import tvm.testing
from tvm.relay import testing
from tvm import relax, relay
from tvm.relax.testing import relay_translator, nn
from tvm.runtime import vm as vm_rt
from tvm.script import relax as R
import numpy as np

[16:58:37] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[16:58:37] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[16:58:37] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[16:58:37] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[16:58:37] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[16:58:37] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled w

In [2]:
builder = relax.BlockBuilder()

# input_size = 784
# hidden_size = 128
# output_size = 10

input_size = 64
hidden_size = 10
output_size = 4

dtype = "float32"

weights_matrix = np.random.random((hidden_size, output_size)).astype(dtype)
bias_matrix = np.random.random((output_size,)).astype(dtype)

with builder.function("main"):
    input = relax.Var("x", R.Tensor((input_size, hidden_size), dtype))
    weights = relax.Constant(tvm.nd.array(weights_matrix))
    bias = relax.Constant(tvm.nd.array(bias_matrix))
    output_matmul = relax.op.matmul(input, weights)
    output_bias = relax.op.add(output_matmul, bias)
    builder.emit_func_output(output_bias, params=[input])

mod = builder.get()
mod.show()

In [3]:
@tvm.instrument.pass_instrument
class MyInstrument:

    def __init__(self):
        self.skip_pass_name = []
        self.output = []
        self.output_after = []
        self.idx = 0

    def run_before_pass(self, mod, pass_info):
        self.idx += 1
        handle = mod.handle
        g = dict(mod.global_var_map_)
        g_ = list(g.keys())
        if len(g_) == 0:
            return
        print(self.idx * "  " + ">", self.idx, pass_info.name, g_, len(self.output))
        # print(dir(mod))
        
        # tmp = (mod.astext(show_meta_data=True), str(pass_info))
        # tmp = (str(mod), str(pass_info))
        tmp = (mod.script(show_meta=True), str(pass_info))
        tmp = (mod, pass_info)
        self.output.append(tmp)


    def run_after_pass(self, mod, pass_info):
        self.idx -= 1
        handle = mod.handle
        g = dict(mod.global_var_map_)
        g_ = list(g.keys())
        if len(g_) == 0:
            return
        print((self.idx + 1) * "  " + "<", self.idx + 1, pass_info.name, g_, len(self.output_after))
        # print(dir(mod))
        # tmp = (mod.astext(show_meta_data=True), str(pass_info))
        tmp = (str(mod), str(pass_info))
        tmp = (mod, pass_info)
        self.output_after.append(tmp)
        pass


In [8]:
# RELAX
my_instrument = MyInstrument()
with tvm.transform.PassContext(instruments=[my_instrument]):
    # ex = relax.build(mod, target=tvm.target.Target("llvm", host="llvm"), pipeline="micro_build", exec_mode="compiled")
    # ex = relax.build(mod, target=tvm.target.Target("llvm", host="llvm"), pipeline="micro2_build", exec_mode="compiled")
    # ex = relax.build(mod, target=tvm.target.Target("c", host="c"), pipeline="micro_build", exec_mode="crt", system_lib=True)
    ex = relax.build(mod, target=tvm.target.Target("c", host="c"), pipeline="micro2_build", exec_mode="crt", system_lib=True)

build
  > 1 _pipeline ['main'] 0
    > 2 sequential ['main'] 1
      > 3 DispatchSortScan ['main'] 2
VisitBinding_ CallNode
        > 4 NormalizeGlobalVar ['main'] 3
        < 4 NormalizeGlobalVar ['main'] 0
      < 3 DispatchSortScan ['main'] 1
      > 3 LegalizeOps ['main'] 4
VisitBinding_ CallNode
VisitBinding_ CallNode
      < 3 LegalizeOps ['main', 'matmul', 'add'] 2
      > 3 RewriteDataflowReshape ['main', 'matmul', 'add'] 5
      < 3 RewriteDataflowReshape ['main', 'matmul', 'add'] 3
      > 3 ToNonDataflow ['main', 'matmul', 'add'] 6
VisitBinding_ CallNode
VisitBinding_ CallNode
      < 3 ToNonDataflow ['main', 'matmul', 'add'] 4
      > 3 RemovePurityChecking ['main', 'matmul', 'add'] 7
VisitBinding_ CallNode
VisitBinding_ CallNode
      < 3 RemovePurityChecking ['main', 'matmul', 'add'] 5
      > 3 CallTIRRewrite ['main', 'matmul', 'add'] 8
VisitBinding_ CallNode
VisitBinding_ CallNode
      < 3 CallTIRRewrite ['main', 'matmul', 'add'] 6
      > 3 ComputePrimValue ['main', '

[17:01:06] /var/tmp/ga87puy/tvm_relax/src/relax/ir/expr_functor.cc:659: VisitBinding_ CallNode

[17:01:06] /var/tmp/ga87puy/tvm_relax/src/relax/ir/expr_functor.cc:659: VisitBinding_ CallNode

[17:01:06] /var/tmp/ga87puy/tvm_relax/src/relax/ir/expr_functor.cc:659: VisitBinding_ CallNode

[17:01:06] /var/tmp/ga87puy/tvm_relax/src/relax/ir/expr_functor.cc:659: VisitBinding_ CallNode

[17:01:06] /var/tmp/ga87puy/tvm_relax/src/relax/ir/expr_functor.cc:659: VisitBinding_ CallNode

[17:01:06] /var/tmp/ga87puy/tvm_relax/src/relax/ir/expr_functor.cc:659: VisitBinding_ CallNode

[17:01:06] /var/tmp/ga87puy/tvm_relax/src/relax/ir/expr_functor.cc:659: VisitBinding_ CallNode

[17:01:06] /var/tmp/ga87puy/tvm_relax/src/relax/ir/expr_functor.cc:659: VisitBinding_ CallNode

[17:01:06] /var/tmp/ga87puy/tvm_relax/src/relax/ir/expr_functor.cc:659: VisitBinding_ CallNode

[17:01:06] /var/tmp/ga87puy/tvm_relax/src/relax/ir/expr_functor.cc:659: VisitBinding_ CallNode

[17:01:07] /var/tmp/ga87puy/tvm_relax/sr

    < 2 tir.TextureFlatten ['add', 'matmul', '__vmtir__main'] 12
    > 2 tir.StorageFlatten ['add', 'matmul', '__vmtir__main'] 14
    < 2 tir.StorageFlatten ['add', 'matmul', '__vmtir__main'] 13
    > 2 tir.LowerCrossThreadReduction ['add', 'matmul', '__vmtir__main'] 15
    < 2 tir.LowerCrossThreadReduction ['add', 'matmul', '__vmtir__main'] 14
    > 2 tir.LowerInitBlock ['add', 'matmul', '__vmtir__main'] 16
    < 2 tir.LowerInitBlock ['add', 'matmul', '__vmtir__main'] 15
    > 2 tir.PlanAndUpdateBufferAllocationLocation ['add', 'matmul', '__vmtir__main'] 17
    < 2 tir.PlanAndUpdateBufferAllocationLocation ['add', 'matmul', '__vmtir__main'] 16
    > 2 tir.ConvertBlocksToOpaque ['add', 'matmul', '__vmtir__main'] 18
    < 2 tir.ConvertBlocksToOpaque ['add', 'matmul', '__vmtir__main'] 17
    > 2 tir.LiftThreadBinding ['add', 'matmul', '__vmtir__main'] 19
    < 2 tir.LiftThreadBinding ['add', 'matmul', '__vmtir__main'] 18
    > 2 tir.ManifestSharedMemoryLocalStage ['add', 'matmul', '__vmt

[17:01:07] /var/tmp/ga87puy/tvm_relax/src/driver/driver_api.cc:568: MixedModulePassManager



    < 2 tir.BindTarget ['add', 'matmul', '__vmtir__main'] 50
    > 2 tir.FP8ComputeLegalize ['add', 'matmul', '__vmtir__main'] 52
    < 2 tir.FP8ComputeLegalize ['add', 'matmul', '__vmtir__main'] 51
    > 2 tir.calculate_allocated_bytes ['add', 'matmul', '__vmtir__main'] 53
    < 2 tir.calculate_allocated_bytes ['add', 'matmul', '__vmtir__main'] 52
    > 2 tir.LowerVtcmAlloc ['add', 'matmul', '__vmtir__main'] 54
    < 2 tir.LowerVtcmAlloc ['add', 'matmul', '__vmtir__main'] 53
    > 2 tir.VerifyMemory ['add', 'matmul', '__vmtir__main'] 55
    < 2 tir.VerifyMemory ['add', 'matmul', '__vmtir__main'] 54
    > 2 tir.AnnotateEntryFunc ['add', 'matmul', '__vmtir__main'] 56
    < 2 tir.AnnotateEntryFunc ['add', 'matmul', '__vmtir__main'] 55
    > 2 tir.ThreadSync ['add', 'matmul', '__vmtir__main'] 57
    < 2 tir.ThreadSync ['add', 'matmul', '__vmtir__main'] 56
    > 2 tir.ThreadSync ['add', 'matmul', '__vmtir__main'] 58
    < 2 tir.ThreadSync ['add', 'matmul', '__vmtir__main'] 57
    > 2 tir.T

[17:01:08] /var/tmp/ga87puy/tvm_relax/src/driver/driver_api.cc:638: HostModulePassManager



    > 2 tir.LowerTVMBuiltin ['add', 'matmul', '__vmtir__main'] 73
    < 2 tir.LowerTVMBuiltin ['add', 'matmul', '__vmtir__main'] 72
    > 2 tir.LowerCustomDatatypes ['add', 'matmul', '__vmtir__main'] 74
    < 2 tir.LowerCustomDatatypes ['add', 'matmul', '__vmtir__main'] 73
    > 2 tir.LowerIntrin ['add', 'matmul', '__vmtir__main'] 75
    < 2 tir.LowerIntrin ['add', 'matmul', '__vmtir__main'] 74
    > 2 tir.LowerDeviceStorageAccessInfo ['add', 'matmul', '__vmtir__main'] 76
    < 2 tir.LowerDeviceStorageAccessInfo ['add', 'matmul', '__vmtir__main'] 75
    > 2 tir.CombineContextCall ['add', 'matmul', '__vmtir__main'] 77
    < 2 tir.CombineContextCall ['add', 'matmul', '__vmtir__main'] 76
  < 1 sequential ['add', 'matmul', '__vmtir__main'] 77
  > 1 sequential ['add', 'matmul', '__vmtir__main'] 78
    > 2 tir.Filter ['add', 'matmul', '__vmtir__main'] 79


[17:01:09] /var/tmp/ga87puy/tvm_relax/src/driver/driver_api.cc:673: DeviceModulePassManager



In [5]:
ex

In [7]:
print(my_instrument.output_after[11][0].show())

None


In [7]:
ex
print(ex.mod._collect_dso_modules()[1].get_source())

// tvm target: c -keys=cpu 
#define TVM_EXPORTS
#include "tvm/runtime/c_runtime_api.h"
#include "tvm/runtime/c_backend_api.h"
#include <math.h>
#include <stdbool.h>

#ifdef __cplusplus
extern "C" {
#endif
static const float __attribute__((section(".rodata.tvm"), aligned(16))) constant_0[40] = {
    0x1.342336p-2, 0x1.eba59ap-5, 0x1.c4f3ecp-1, 0x1.093d32p-1, 0x1.4db1aep-3, 0x1.743d7ap-2, 0x1.41134cp-1, 0x1.d41f26p-1, 
    0x1.45bc2ep-5, 0x1.41f6f8p-10, 0x1.c22482p-2, 0x1.92113ap-2, 0x1.0f10ccp-5, 0x1.cb506ap-3, 0x1.f4ecdcp-1, 0x1.3940dep-2, 
    0x1.834daep-1, 0x1.796a6p-3, 0x1.e7c10cp-1, 0x1.952422p-2, 0x1.a749aap-4, 0x1.94d0ap-1, 0x1.b541a8p-1, 0x1.6effb6p-1, 
    0x1.a83612p-5, 0x1.6d474ap-2, 0x1.d52f1ap-1, 0x1.ecb864p-2, 0x1.f295fcp-5, 0x1.a74f98p-5, 0x1.cc73b4p-3, 0x1.afb60ep-1, 
    0x1.9d85bcp-3, 0x1.29fa0ep-1, 0x1.e42c92p-1, 0x1.85c522p-1, 0x1.24dc56p-1, 0x1.735c1ap-2, 0x1.e32908p-1, 0x1.50f64cp-1
};
#ifdef __cplusplus
}  // extern "C"
#endif

#ifdef __cplusplus
extern "C" {
#en

In [8]:
# NON-RELAX
my_instrument = MyInstrument()
with tvm.transform.PassContext(instruments=[my_instrument], config={"tir.usmp.enable": False, "relay.FuseOps.max_depth": 1, "tir.disable_vectorize": True}):
    ex = relay.build(conv2d_mod, target=tvm.target.Target("c", host="c"), runtime=tvm.relay.backend.Runtime("crt"), executor=tvm.relay.backend.Executor("aot", {"interface-api": "packed", "unpacked-api": False}))

NameError: name 'conv2d_mod' is not defined

In [14]:
print(ex.module._collect_dso_modules()[2].get_source())


// tvm target: c -keys=cpu 
#define TVM_EXPORTS
#include "tvm/runtime/c_runtime_api.h"
#include "tvm/runtime/c_backend_api.h"
#include <math.h>
#include <stdbool.h>

#ifdef __cplusplus
extern "C" {
#endif
static const int32_t __attribute__((section(".rodata.tvm"), aligned(16))) fused_constant_1[10] = {
    +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, 
    +0x00000000, +0x00000000
};
#ifdef __cplusplus
}  // extern "C"
#endif

#ifdef __cplusplus
extern "C" {
#endif
static const int32_t __attribute__((section(".rodata.tvm"), aligned(16))) fused_constant[1280] = {
    +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, 
    +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, 
    +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, +0x00000000, 
    +0x00000000, +0x00000000, +0x00000000, +0x

In [17]:
print(my_instrument.output_after[190][0])
# print(my_instrument.output_after[14][0])

def @main(%x {virtual_device=VirtualDevice(device_type=1, virtual_device_id=0, target=Target(id=2a1a0a0, kind='c', keys={'cpu'}, host=Target(id=299ea40, kind='c', keys={'cpu'})))}: Tensor[(784, 128), int32] /* ty=Tensor[(784, 128), int32] */, executor=meta[Executor][0], runtime=meta[Runtime][0], hash="7e7f4fb496fdac41", io_used_memory=432768, virtual_device=VirtualDevice(device_type=1, virtual_device_id=0, target=Target(id=2a1a0a0, kind='c', keys={'cpu'}, host=Target(id=299ea40, kind='c', keys={'cpu'})))) -> Tensor[(784, 10), int32] {
  %0 = (%x,) /* ty=(Tensor[(784, 128), int32],) */;
  %1 = call_lowered(@tvmgen_default_fused_nn_matmul, %0, metadata={"relay_attrs"={__dict__={"Primitive"=1, "hash"="c448be2e2f295841", "used_memory"=[432768]}}, "all_prim_fn_vars"=['tvmgen_default_fused_nn_matmul']}) /* ty=Tensor[(784, 10), int32] */;
  let %x_5: Tensor[(784, 10), int32] /* ty=Tensor[(784, 10), int32] */ = on_device(%1, virtual_device=VirtualDevice(device_type=1, virtual_device_id=0, targ

In [13]:
dir(ex)
111

111

In [33]:
ex.as_text()

''

In [34]:
ex.as_python()

'ib = rx.Builder()\n'

In [12]:
input_size = 784
hidden_size = 128
output_size = 10

dtype = "int32" # float32

weights_matrix = np.random.random((hidden_size, output_size)).astype(dtype)
bias_matrix = np.random.random((output_size,)).astype(dtype)

def relay_dense():
    """
    Simple dense Relay implementation.
    """
    dtype = "float32"

    x = relay.var("x", shape=(input_size, hidden_size), dtype=dtype)
    weight = relay.const(tvm.nd.array(weights_matrix))
    bias = relay.const(tvm.nd.array(bias_matrix))
    output_matmul = relay.nn.matmul(x, weight)
    output_bias = relay.op.add(output_matmul, bias)
    func = relay.Function(relay.analysis.free_vars(output_bias), output_bias)
    return tvm.IRModule.from_expr(func)
    return func
dense_mod = relay_dense()

def relay_conv2d():
    """
    Simple conv2d Relay implementation.
    """
    dtype = "float32"
    # TODO

    x = relay.var("x", shape=(input_size, hidden_size), dtype=dtype)
    weight = relay.const(tvm.nd.array(weights_matrix))
    bias = relay.const(tvm.nd.array(bias_matrix))
    output_matmul = relay.nn.matmul(x, weight)
    output_bias = relay.op.add(output_matmul, bias)
    func = relay.Function(relay.analysis.free_vars(output_bias), output_bias)
    return tvm.IRModule.from_expr(func)
    return func
conv2d_mod = relay_conv2d()

In [13]:
conv2d_mod

def @main(%x: Tensor[(784, 128), int32]) {
  %0 = nn.matmul(%x, meta[relay.Constant][0], units=None);
  add(%0, meta[relay.Constant][1])
}


In [26]:
conv2d_mod.show()

In [3]:
from tvm.script import ir as I
from tvm.script import tir as T

@I.ir_module
class Module:
    I.module_attrs({"runtime": None})
    @T.prim_func
    def main(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""})})
        assert num_args == 4, "__vmtir__main: num_args should be 4"
        assert not T.isnullptr(args), "__vmtir__main: TVMValue* arg pointer was NULL"
        assert not T.isnullptr(arg_type_ids), "__vmtir__main: int* type_codes was NULL"
        arg_type_ids_1 = T.decl_buffer((4,), "int32", data=arg_type_ids)
        ctx_ptr_code: T.int32 = arg_type_ids_1[0]
        assert ctx_ptr_code == 3 or ctx_ptr_code == 13 or ctx_ptr_code == 7 or ctx_ptr_code == 4, "__vmtir__main: Expect arg[0] to be pointer"
        r_code: T.int32 = arg_type_ids_1[1]
        assert r_code == 3 or r_code == 13 or r_code == 7 or r_code == 4, "__vmtir__main: Expect arg[1] to be pointer"
        c_code: T.int32 = arg_type_ids_1[2]
        assert c_code == 3 or c_code == 13 or c_code == 7 or c_code == 4, "__vmtir__main: Expect arg[2] to be pointer"
        f_code: T.int32 = arg_type_ids_1[3]
        assert f_code == 3 or f_code == 13 or f_code == 7 or f_code == 4, "__vmtir__main: Expect arg[3] to be pointer"
        ctx_ptr: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        r: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        c: T.handle = T.tvm_struct_get(args, 2, 12, "handle")
        f: T.handle = T.tvm_struct_get(args, 3, 12, "handle")
        with T.attr(0, "compute_scope", "__vmtir__main_compute_"):
            T.anylist_setitem_call_packed(r, 2, "vm.builtin.alloc_storage", ctx_ptr, T.anylist_getitem(c, 0), T.int64(0), T.anylist_getitem(c, 1), T.anylist_getitem(c, 2))
            T.anylist_setitem_call_packed(r, 3, "vm.builtin.alloc_tensor", T.anylist_getitem(r, 2), T.int64(0), T.anylist_getitem(c, 3), T.anylist_getitem(c, 4))
            T.anylist_setitem_call_packed(r, 2, "vm.builtin.null_value")
            T.call_cpacked("matmul", T.anylist_getitem(r, 0), T.anylist_getitem(c, 5), T.anylist_getitem(r, 3), T.reinterpret("handle", T.uint64(0)))
            T.anylist_setitem_call_packed(r, 4, "vm.builtin.alloc_storage", ctx_ptr, T.anylist_getitem(c, 0), T.int64(0), T.anylist_getitem(c, 1), T.anylist_getitem(c, 6))
            T.anylist_setitem_call_packed(r, 5, "vm.builtin.alloc_tensor", T.anylist_getitem(r, 4), T.int64(0), T.anylist_getitem(c, 3), T.anylist_getitem(c, 7))
            T.anylist_setitem_call_packed(r, 4, "vm.builtin.null_value")
            T.call_cpacked("add", T.anylist_getitem(r, 3), T.anylist_getitem(c, 8), T.anylist_getitem(r, 5), T.reinterpret("handle", T.uint64(0)))
            T.anylist_setitem_call_packed(r, 3, "vm.builtin.null_value")
            T.anylist_setitem_call_packed(r, 1, "vm.builtin.copy", T.anylist_getitem(r, 5))
        return 0

    @T.prim_func
    def add(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""}), "tir.noalias": T.bool(True)})
        assert num_args == 3, "add: num_args should be 3"
        assert not T.isnullptr(args), "add: TVMValue* arg pointer was NULL"
        assert not T.isnullptr(arg_type_ids), "add: int* type_codes was NULL"
        arg_type_ids_1 = T.decl_buffer((3,), "int32", data=arg_type_ids)
        var_A_code: T.int32 = arg_type_ids_1[0]
        assert var_A_code == 3 or var_A_code == 13 or var_A_code == 7 or var_A_code == 4, "add: Expect arg[0] to be pointer"
        var_B_code: T.int32 = arg_type_ids_1[1]
        assert var_B_code == 3 or var_B_code == 13 or var_B_code == 7 or var_B_code == 4, "add: Expect arg[1] to be pointer"
        var_T_add_code: T.int32 = arg_type_ids_1[2]
        assert var_T_add_code == 3 or var_T_add_code == 13 or var_T_add_code == 7 or var_T_add_code == 4, "add: Expect arg[2] to be pointer"
        var_A: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        var_B: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        var_T_add: T.handle = T.tvm_struct_get(args, 2, 12, "handle")
        assert not T.isnullptr(var_A), "add.var_A is expected to have non-NULL DLTensor* pointer"
        assert 2 == T.tvm_struct_get(var_A, 0, 4, "int32"), "add.var_A.ndim is expected to equal 2"
        add_var_A_shape: T.handle("int64") = T.tvm_struct_get(var_A, 0, 2, "handle")
        add_var_A_shape_1 = T.decl_buffer((2,), "int64", data=add_var_A_shape)
        add_var_A_strides: T.handle("int64") = T.tvm_struct_get(var_A, 0, 3, "handle")
        add_var_A_strides_1 = T.decl_buffer((0,), "int64", data=add_var_A_strides)
        dev_id: T.int32 = T.tvm_struct_get(var_A, 0, 9, "int32")
        A: T.handle("float32", "global") = T.tvm_struct_get(var_A, 0, 1, "handle")
        T.attr(A, "storage_alignment", 64)
        assert not T.isnullptr(var_B), "add.var_B is expected to have non-NULL DLTensor* pointer"
        assert 1 == T.tvm_struct_get(var_B, 0, 4, "int32"), "add.var_B.ndim is expected to equal 1"
        add_var_B_shape: T.handle("int64") = T.tvm_struct_get(var_B, 0, 2, "handle")
        add_var_B_shape_1 = T.decl_buffer((1,), "int64", data=add_var_B_shape)
        add_var_B_strides: T.handle("int64") = T.tvm_struct_get(var_B, 0, 3, "handle")
        add_var_B_strides_1 = T.decl_buffer((0,), "int64", data=add_var_B_strides)
        B: T.handle("float32", "global") = T.tvm_struct_get(var_B, 0, 1, "handle")
        T.attr(B, "storage_alignment", 64)
        assert not T.isnullptr(var_T_add), "add.var_T_add is expected to have non-NULL DLTensor* pointer"
        assert 2 == T.tvm_struct_get(var_T_add, 0, 4, "int32"), "add.var_T_add.ndim is expected to equal 2"
        add_var_T_add_shape: T.handle("int64") = T.tvm_struct_get(var_T_add, 0, 2, "handle")
        add_var_T_add_shape_1 = T.decl_buffer((2,), "int64", data=add_var_T_add_shape)
        add_var_T_add_strides: T.handle("int64") = T.tvm_struct_get(var_T_add, 0, 3, "handle")
        add_var_T_add_strides_1 = T.decl_buffer((0,), "int64", data=add_var_T_add_strides)
        T_add: T.handle("float32", "global") = T.tvm_struct_get(var_T_add, 0, 1, "handle")
        T.attr(T_add, "storage_alignment", 64)
        T.attr("default", "device_id", dev_id)
        T.attr("default", "device_type", 1)
        assert T.tvm_struct_get(var_A, 0, 5, "uint8") == T.uint8(2) and T.tvm_struct_get(var_A, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_A, 0, 7, "uint16") == T.uint16(1), "add.var_A.dtype is expected to be float32"
        assert add_var_A_shape_1[0] == T.int64(784), "Argument add.var_A.shape[0] has an unsatisfied constraint: T.int64(784) == add_var_A_shape[0]"
        assert add_var_A_shape_1[1] == T.int64(10), "Argument add.var_A.shape[1] has an unsatisfied constraint: T.int64(10) == add_var_A_shape[1]"
        if not T.isnullptr(add_var_A_strides):
            assert T.int64(1) == add_var_A_strides_1[1] and T.int64(10) == add_var_A_strides_1[0], "add.var_A.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_A, 0, 8, "uint64"), "Argument add.var_A.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_A, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_A, 0, 10, "int32") == 1, "Argument add.var_A.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_A, 0, 10, \"int32\")"
        assert not T.isnullptr(A), "add.var_A is expected to have non-NULL data pointer"
        assert T.tvm_struct_get(var_B, 0, 5, "uint8") == T.uint8(2) and T.tvm_struct_get(var_B, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_B, 0, 7, "uint16") == T.uint16(1), "add.var_B.dtype is expected to be float32"
        assert add_var_B_shape_1[0] == T.int64(10), "Argument add.var_B.shape[0] has an unsatisfied constraint: T.int64(10) == add_var_B_shape[0]"
        if not T.isnullptr(add_var_B_strides):
            assert T.int64(1) == add_var_B_strides_1[0], "add.var_B.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_B, 0, 8, "uint64"), "Argument add.var_B.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_B, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_B, 0, 10, "int32") == 1, "Argument add.var_B.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_B, 0, 10, \"int32\")"
        assert dev_id == T.tvm_struct_get(var_B, 0, 9, "int32"), "Argument add.var_B.device_id has an unsatisfied constraint: dev_id == T.tvm_struct_get(var_B, 0, 9, \"int32\")"
        assert not T.isnullptr(B), "add.var_B is expected to have non-NULL data pointer"
        assert T.tvm_struct_get(var_T_add, 0, 5, "uint8") == T.uint8(2) and T.tvm_struct_get(var_T_add, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_T_add, 0, 7, "uint16") == T.uint16(1), "add.var_T_add.dtype is expected to be float32"
        assert add_var_T_add_shape_1[0] == T.int64(784), "Argument add.var_T_add.shape[0] has an unsatisfied constraint: T.int64(784) == add_var_T_add_shape[0]"
        assert add_var_T_add_shape_1[1] == T.int64(10), "Argument add.var_T_add.shape[1] has an unsatisfied constraint: T.int64(10) == add_var_T_add_shape[1]"
        if not T.isnullptr(add_var_T_add_strides):
            assert T.int64(1) == add_var_T_add_strides_1[1] and T.int64(10) == add_var_T_add_strides_1[0], "add.var_T_add.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_T_add, 0, 8, "uint64"), "Argument add.var_T_add.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_T_add, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_T_add, 0, 10, "int32") == 1, "Argument add.var_T_add.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_T_add, 0, 10, \"int32\")"
        assert dev_id == T.tvm_struct_get(var_T_add, 0, 9, "int32"), "Argument add.var_T_add.device_id has an unsatisfied constraint: dev_id == T.tvm_struct_get(var_T_add, 0, 9, \"int32\")"
        assert not T.isnullptr(T_add), "add.var_T_add is expected to have non-NULL data pointer"
        A_1 = T.decl_buffer((T.int64(784), T.int64(10)), data=A)
        B_1 = T.decl_buffer((T.int64(10),), data=B)
        T_add_1 = T.decl_buffer((T.int64(784), T.int64(10)), data=T_add)
        with T.attr(0, "compute_scope", "add_compute_"):
            for ax0, ax1 in T.grid(784, 10):
                cse_var_1: T.int32 = ax0 * 10 + ax1
                T_add_2 = T.Buffer((T.int64(7840),), data=T_add)
                A_2 = T.Buffer((T.int64(7840),), data=A)
                B_2 = T.Buffer((T.int64(10),), data=B)
                T_add_2[cse_var_1] = A_2[cse_var_1] + B_2[ax1]
        return 0

    @T.prim_func
    def matmul(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""}), "tir.noalias": T.bool(True)})
        assert num_args == 3, "matmul: num_args should be 3"
        assert not T.isnullptr(args), "matmul: TVMValue* arg pointer was NULL"
        assert not T.isnullptr(arg_type_ids), "matmul: int* type_codes was NULL"
        arg_type_ids_1 = T.decl_buffer((3,), "int32", data=arg_type_ids)
        var_A_code: T.int32 = arg_type_ids_1[0]b
        assert var_A_code == 3 or var_A_code == 13 or var_A_code == 7 or var_A_code == 4, "matmul: Expect arg[0] to be pointer"
        var_B_code: T.int32 = arg_type_ids_1[1]
        assert var_B_code == 3 or var_B_code == 13 or var_B_code == 7 or var_B_code == 4, "matmul: Expect arg[1] to be pointer"
        var_matmul_code: T.int32 = arg_type_ids_1[2]
        assert var_matmul_code == 3 or var_matmul_code == 13 or var_matmul_code == 7 or var_matmul_code == 4, "matmul: Expect arg[2] to be pointer"
        var_A: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        var_B: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        var_matmul: T.handle = T.tvm_struct_get(args, 2, 12, "handle")
        assert not T.isnullptr(var_A), "matmul.var_A is expected to have non-NULL DLTensor* pointer"
        assert 2 == T.tvm_struct_get(var_A, 0, 4, "int32"), "matmul.var_A.ndim is expected to equal 2"
        matmul_var_A_shape: T.handle("int64") = T.tvm_struct_get(var_A, 0, 2, "handle")
        matmul_var_A_shape_1 = T.decl_buffer((2,), "int64", data=matmul_var_A_shape)
        matmul_var_A_strides: T.handle("int64") = T.tvm_struct_get(var_A, 0, 3, "handle")
        matmul_var_A_strides_1 = T.decl_buffer((0,), "int64", data=matmul_var_A_strides)
        dev_id: T.int32 = T.tvm_struct_get(var_A, 0, 9, "int32")
        A: T.handle("float32", "global") = T.tvm_struct_get(var_A, 0, 1, "handle")
        T.attr(A, "storage_alignment", 64)
        assert not T.isnullptr(var_B), "matmul.var_B is expected to have non-NULL DLTensor* pointer"
        assert 2 == T.tvm_struct_get(var_B, 0, 4, "int32"), "matmul.var_B.ndim is expected to equal 2"
        matmul_var_B_shape: T.handle("int64") = T.tvm_struct_get(var_B, 0, 2, "handle")
        matmul_var_B_shape_1 = T.decl_buffer((2,), "int64", data=matmul_var_B_shape)
        matmul_var_B_strides: T.handle("int64") = T.tvm_struct_get(var_B, 0, 3, "handle")
        matmul_var_B_strides_1 = T.decl_buffer((0,), "int64", data=matmul_var_B_strides)
        B: T.handle("float32", "global") = T.tvm_struct_get(var_B, 0, 1, "handle")
        T.attr(B, "storage_alignment", 64)
        assert not T.isnullptr(var_matmul), "matmul.var_matmul is expected to have non-NULL DLTensor* pointer"
        assert 2 == T.tvm_struct_get(var_matmul, 0, 4, "int32"), "matmul.var_matmul.ndim is expected to equal 2"
        matmul_var_matmul_shape: T.handle("int64") = T.tvm_struct_get(var_matmul, 0, 2, "handle")
        matmul_var_matmul_shape_1 = T.decl_buffer((2,), "int64", data=matmul_var_matmul_shape)
        matmul_var_matmul_strides: T.handle("int64") = T.tvm_struct_get(var_matmul, 0, 3, "handle")
        matmul_var_matmul_strides_1 = T.decl_buffer((0,), "int64", data=matmul_var_matmul_strides)
        matmul: T.handle("float32", "global") = T.tvm_struct_get(var_matmul, 0, 1, "handle")
        T.attr(matmul, "storage_alignment", 64)
        T.attr("default", "device_id", dev_id)
        T.attr("default", "device_type", 1)
        assert T.tvm_struct_get(var_A, 0, 5, "uint8") == T.uint8(2) and T.tvm_struct_get(var_A, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_A, 0, 7, "uint16") == T.uint16(1), "matmul.var_A.dtype is expected to be float32"
        assert matmul_var_A_shape_1[0] == T.int64(784), "Argument matmul.var_A.shape[0] has an unsatisfied constraint: T.int64(784) == matmul_var_A_shape[0]"
        assert matmul_var_A_shape_1[1] == T.int64(128), "Argument matmul.var_A.shape[1] has an unsatisfied constraint: T.int64(128) == matmul_var_A_shape[1]"
        if not T.isnullptr(matmul_var_A_strides):
            assert T.int64(1) == matmul_var_A_strides_1[1] and T.int64(128) == matmul_var_A_strides_1[0], "matmul.var_A.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_A, 0, 8, "uint64"), "Argument matmul.var_A.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_A, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_A, 0, 10, "int32") == 1, "Argument matmul.var_A.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_A, 0, 10, \"int32\")"
        assert not T.isnullptr(A), "matmul.var_A is expected to have non-NULL data pointer"
        assert T.tvm_struct_get(var_B, 0, 5, "uint8") == T.uint8(2) and T.tvm_struct_get(var_B, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_B, 0, 7, "uint16") == T.uint16(1), "matmul.var_B.dtype is expected to be float32"
        assert matmul_var_B_shape_1[0] == T.int64(128), "Argument matmul.var_B.shape[0] has an unsatisfied constraint: T.int64(128) == matmul_var_B_shape[0]"
        assert matmul_var_B_shape_1[1] == T.int64(10), "Argument matmul.var_B.shape[1] has an unsatisfied constraint: T.int64(10) == matmul_var_B_shape[1]"
        if not T.isnullptr(matmul_var_B_strides):
            assert T.int64(1) == matmul_var_B_strides_1[1] and T.int64(10) == matmul_var_B_strides_1[0], "matmul.var_B.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_B, 0, 8, "uint64"), "Argument matmul.var_B.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_B, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_B, 0, 10, "int32") == 1, "Argument matmul.var_B.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_B, 0, 10, \"int32\")"
        assert dev_id == T.tvm_struct_get(var_B, 0, 9, "int32"), "Argument matmul.var_B.device_id has an unsatisfied constraint: dev_id == T.tvm_struct_get(var_B, 0, 9, \"int32\")"
        assert not T.isnullptr(B), "matmul.var_B is expected to have non-NULL data pointer"
        assert T.tvm_struct_get(var_matmul, 0, 5, "uint8") == T.uint8(2) and T.tvm_struct_get(var_matmul, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_matmul, 0, 7, "uint16") == T.uint16(1), "matmul.var_matmul.dtype is expected to be float32"
        assert matmul_var_matmul_shape_1[0] == T.int64(784), "Argument matmul.var_matmul.shape[0] has an unsatisfied constraint: T.int64(784) == matmul_var_matmul_shape[0]"
        assert matmul_var_matmul_shape_1[1] == T.int64(10), "Argument matmul.var_matmul.shape[1] has an unsatisfied constraint: T.int64(10) == matmul_var_matmul_shape[1]"
        if not T.isnullptr(matmul_var_matmul_strides):
            assert T.int64(1) == matmul_var_matmul_strides_1[1] and T.int64(10) == matmul_var_matmul_strides_1[0], "matmul.var_matmul.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_matmul, 0, 8, "uint64"), "Argument matmul.var_matmul.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_matmul, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_matmul, 0, 10, "int32") == 1, "Argument matmul.var_matmul.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_matmul, 0, 10, \"int32\")"
        assert dev_id == T.tvm_struct_get(var_matmul, 0, 9, "int32"), "Argument matmul.var_matmul.device_id has an unsatisfied constraint: dev_id == T.tvm_struct_get(var_matmul, 0, 9, \"int32\")"
        assert not T.isnullptr(matmul), "matmul.var_matmul is expected to have non-NULL data pointer"
        A_1 = T.decl_buffer((T.int64(784), T.int64(128)), data=A)
        B_1 = T.decl_buffer((T.int64(128), T.int64(10)), data=B)
        matmul_1 = T.decl_buffer((T.int64(784), T.int64(10)), data=matmul)
        with T.attr(0, "compute_scope", "matmul_compute_"):
            for i0, i1, k in T.grid(784, 10, 128):
                cse_var_1: T.int32 = i0 * 10 + i1
                matmul_2 = T.Buffer((T.int64(7840),), data=matmul)
                if k == 0:
                    matmul_2[cse_var_1] = T.float32(0)
                A_2 = T.Buffer((T.int64(100352),), data=A)
                B_2 = T.Buffer((T.int64(1280),), data=B)
                matmul_2[cse_var_1] = matmul_2[cse_var_1] + A_2[i0 * 128 + k] * B_2[k * 10 + i1]
        return 0

In [9]:
from tvm.script import ir as I
from tvm.script import tir as T

@I.ir_module
class Module:
    I.module_attrs({"runtime": None})
    @T.prim_func
    def __vmtir__main(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""})})
        assert num_args == 4, "__vmtir__main: num_args should be 4"
        assert not T.isnullptr(args), "__vmtir__main: TVMValue* arg pointer was NULL"
        assert not T.isnullptr(arg_type_ids), "__vmtir__main: int* type_codes was NULL"
        arg_type_ids_1 = T.decl_buffer((4,), "int32", data=arg_type_ids)
        ctx_ptr_code: T.int32 = arg_type_ids_1[0]
        assert ctx_ptr_code == 3 or ctx_ptr_code == 13 or ctx_ptr_code == 7 or ctx_ptr_code == 4, "__vmtir__main: Expect arg[0] to be pointer"
        r_code: T.int32 = arg_type_ids_1[1]
        assert r_code == 3 or r_code == 13 or r_code == 7 or r_code == 4, "__vmtir__main: Expect arg[1] to be pointer"
        c_code: T.int32 = arg_type_ids_1[2]
        assert c_code == 3 or c_code == 13 or c_code == 7 or c_code == 4, "__vmtir__main: Expect arg[2] to be pointer"
        f_code: T.int32 = arg_type_ids_1[3]
        assert f_code == 3 or f_code == 13 or f_code == 7 or f_code == 4, "__vmtir__main: Expect arg[3] to be pointer"
        ctx_ptr: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        r: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        c: T.handle = T.tvm_struct_get(args, 2, 12, "handle")
        f: T.handle = T.tvm_struct_get(args, 3, 12, "handle")
        with T.attr(0, "compute_scope", "__vmtir__main_compute_"):
            T.anylist_setitem_call_packed(r, 2, "vm.builtin.alloc_storage", ctx_ptr, T.anylist_getitem(c, 0), T.int64(0), T.anylist_getitem(c, 1), T.anylist_getitem(c, 2))
            T.anylist_setitem_call_packed(r, 3, "vm.builtin.alloc_tensor", T.anylist_getitem(r, 2), T.int64(0), T.anylist_getitem(c, 3), T.anylist_getitem(c, 4))
            T.anylist_setitem_call_packed(r, 2, "vm.builtin.null_value")
            T.call_cpacked("matmul", T.anylist_getitem(r, 0), T.anylist_getitem(c, 5), T.anylist_getitem(r, 3), T.reinterpret("handle", T.uint64(0)))
            T.anylist_setitem_call_packed(r, 4, "vm.builtin.alloc_storage", ctx_ptr, T.anylist_getitem(c, 0), T.int64(0), T.anylist_getitem(c, 1), T.anylist_getitem(c, 6))
            T.anylist_setitem_call_packed(r, 5, "vm.builtin.alloc_tensor", T.anylist_getitem(r, 4), T.int64(0), T.anylist_getitem(c, 3), T.anylist_getitem(c, 7))
            T.anylist_setitem_call_packed(r, 4, "vm.builtin.null_value")
            T.call_cpacked("add", T.anylist_getitem(r, 3), T.anylist_getitem(c, 8), T.anylist_getitem(r, 5), T.reinterpret("handle", T.uint64(0)))
            T.anylist_setitem_call_packed(r, 3, "vm.builtin.null_value")
            T.anylist_setitem_call_packed(r, 1, "vm.builtin.copy", T.anylist_getitem(r, 5))
        return 0

    @T.prim_func
    def add(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""}), "tir.noalias": T.bool(True)})
        assert num_args == 3, "add: num_args should be 3"
        assert not T.isnullptr(args), "add: TVMValue* arg pointer was NULL"
        assert not T.isnullptr(arg_type_ids), "add: int* type_codes was NULL"
        arg_type_ids_1 = T.decl_buffer((3,), "int32", data=arg_type_ids)
        var_A_code: T.int32 = arg_type_ids_1[0]
        assert var_A_code == 3 or var_A_code == 13 or var_A_code == 7 or var_A_code == 4, "add: Expect arg[0] to be pointer"
        var_B_code: T.int32 = arg_type_ids_1[1]
        assert var_B_code == 3 or var_B_code == 13 or var_B_code == 7 or var_B_code == 4, "add: Expect arg[1] to be pointer"
        var_T_add_code: T.int32 = arg_type_ids_1[2]
        assert var_T_add_code == 3 or var_T_add_code == 13 or var_T_add_code == 7 or var_T_add_code == 4, "add: Expect arg[2] to be pointer"
        var_A: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        var_B: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        var_T_add: T.handle = T.tvm_struct_get(args, 2, 12, "handle")
        assert not T.isnullptr(var_A), "add.var_A is expected to have non-NULL DLTensor* pointer"
        assert 2 == T.tvm_struct_get(var_A, 0, 4, "int32"), "add.var_A.ndim is expected to equal 2"
        add_var_A_shape: T.handle("int64") = T.tvm_struct_get(var_A, 0, 2, "handle")
        add_var_A_shape_1 = T.decl_buffer((2,), "int64", data=add_var_A_shape)
        add_var_A_strides: T.handle("int64") = T.tvm_struct_get(var_A, 0, 3, "handle")
        add_var_A_strides_1 = T.decl_buffer((0,), "int64", data=add_var_A_strides)
        dev_id: T.int32 = T.tvm_struct_get(var_A, 0, 9, "int32")
        A: T.handle("int32", "global") = T.tvm_struct_get(var_A, 0, 1, "handle")
        T.attr(A, "storage_alignment", 64)
        assert not T.isnullptr(var_B), "add.var_B is expected to have non-NULL DLTensor* pointer"
        assert 1 == T.tvm_struct_get(var_B, 0, 4, "int32"), "add.var_B.ndim is expected to equal 1"
        add_var_B_shape: T.handle("int64") = T.tvm_struct_get(var_B, 0, 2, "handle")
        add_var_B_shape_1 = T.decl_buffer((1,), "int64", data=add_var_B_shape)
        add_var_B_strides: T.handle("int64") = T.tvm_struct_get(var_B, 0, 3, "handle")
        add_var_B_strides_1 = T.decl_buffer((0,), "int64", data=add_var_B_strides)
        B: T.handle("int32", "global") = T.tvm_struct_get(var_B, 0, 1, "handle")
        T.attr(B, "storage_alignment", 64)
        assert not T.isnullptr(var_T_add), "add.var_T_add is expected to have non-NULL DLTensor* pointer"
        assert 2 == T.tvm_struct_get(var_T_add, 0, 4, "int32"), "add.var_T_add.ndim is expected to equal 2"
        add_var_T_add_shape: T.handle("int64") = T.tvm_struct_get(var_T_add, 0, 2, "handle")
        add_var_T_add_shape_1 = T.decl_buffer((2,), "int64", data=add_var_T_add_shape)
        add_var_T_add_strides: T.handle("int64") = T.tvm_struct_get(var_T_add, 0, 3, "handle")
        add_var_T_add_strides_1 = T.decl_buffer((0,), "int64", data=add_var_T_add_strides)
        T_add: T.handle("int32", "global") = T.tvm_struct_get(var_T_add, 0, 1, "handle")
        T.attr(T_add, "storage_alignment", 64)
        T.attr("default", "device_id", dev_id)
        T.attr("default", "device_type", 1)
        assert T.tvm_struct_get(var_A, 0, 5, "uint8") == T.uint8(0) and T.tvm_struct_get(var_A, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_A, 0, 7, "uint16") == T.uint16(1), "add.var_A.dtype is expected to be int32"
        assert add_var_A_shape_1[0] == T.int64(784), "Argument add.var_A.shape[0] has an unsatisfied constraint: T.int64(784) == add_var_A_shape[0]"
        assert add_var_A_shape_1[1] == T.int64(10), "Argument add.var_A.shape[1] has an unsatisfied constraint: T.int64(10) == add_var_A_shape[1]"
        if not T.isnullptr(add_var_A_strides):
            assert T.int64(1) == add_var_A_strides_1[1] and T.int64(10) == add_var_A_strides_1[0], "add.var_A.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_A, 0, 8, "uint64"), "Argument add.var_A.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_A, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_A, 0, 10, "int32") == 1, "Argument add.var_A.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_A, 0, 10, \"int32\")"
        assert not T.isnullptr(A), "add.var_A is expected to have non-NULL data pointer"
        assert T.tvm_struct_get(var_B, 0, 5, "uint8") == T.uint8(0) and T.tvm_struct_get(var_B, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_B, 0, 7, "uint16") == T.uint16(1), "add.var_B.dtype is expected to be int32"
        assert add_var_B_shape_1[0] == T.int64(10), "Argument add.var_B.shape[0] has an unsatisfied constraint: T.int64(10) == add_var_B_shape[0]"
        if not T.isnullptr(add_var_B_strides):
            assert T.int64(1) == add_var_B_strides_1[0], "add.var_B.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_B, 0, 8, "uint64"), "Argument add.var_B.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_B, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_B, 0, 10, "int32") == 1, "Argument add.var_B.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_B, 0, 10, \"int32\")"
        assert dev_id == T.tvm_struct_get(var_B, 0, 9, "int32"), "Argument add.var_B.device_id has an unsatisfied constraint: dev_id == T.tvm_struct_get(var_B, 0, 9, \"int32\")"
        assert not T.isnullptr(B), "add.var_B is expected to have non-NULL data pointer"
        assert T.tvm_struct_get(var_T_add, 0, 5, "uint8") == T.uint8(0) and T.tvm_struct_get(var_T_add, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_T_add, 0, 7, "uint16") == T.uint16(1), "add.var_T_add.dtype is expected to be int32"
        assert add_var_T_add_shape_1[0] == T.int64(784), "Argument add.var_T_add.shape[0] has an unsatisfied constraint: T.int64(784) == add_var_T_add_shape[0]"
        assert add_var_T_add_shape_1[1] == T.int64(10), "Argument add.var_T_add.shape[1] has an unsatisfied constraint: T.int64(10) == add_var_T_add_shape[1]"
        if not T.isnullptr(add_var_T_add_strides):
            assert T.int64(1) == add_var_T_add_strides_1[1] and T.int64(10) == add_var_T_add_strides_1[0], "add.var_T_add.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_T_add, 0, 8, "uint64"), "Argument add.var_T_add.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_T_add, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_T_add, 0, 10, "int32") == 1, "Argument add.var_T_add.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_T_add, 0, 10, \"int32\")"
        assert dev_id == T.tvm_struct_get(var_T_add, 0, 9, "int32"), "Argument add.var_T_add.device_id has an unsatisfied constraint: dev_id == T.tvm_struct_get(var_T_add, 0, 9, \"int32\")"
        assert not T.isnullptr(T_add), "add.var_T_add is expected to have non-NULL data pointer"
        A_1 = T.decl_buffer((T.int64(784), T.int64(10)), "int32", data=A)
        B_1 = T.decl_buffer((T.int64(10),), "int32", data=B)
        T_add_1 = T.decl_buffer((T.int64(784), T.int64(10)), "int32", data=T_add)
        with T.attr(0, "compute_scope", "add_compute_"):
            for ax0, ax1 in T.grid(784, 10):
                cse_var_1: T.int32 = ax0 * 10 + ax1
                T_add_2 = T.Buffer((T.int64(7840),), "int32", data=T_add)
                A_2 = T.Buffer((T.int64(7840),), "int32", data=A)
                B_2 = T.Buffer((T.int64(10),), "int32", data=B)
                T_add_2[cse_var_1] = A_2[cse_var_1] + B_2[ax1]
        return 0

    @T.prim_func
    def matmul(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""}), "tir.noalias": T.bool(True)})
        assert num_args == 3, "matmul: num_args should be 3"
        assert not T.isnullptr(args), "matmul: TVMValue* arg pointer was NULL"
        assert not T.isnullptr(arg_type_ids), "matmul: int* type_codes was NULL"
        arg_type_ids_1 = T.decl_buffer((3,), "int32", data=arg_type_ids)
        var_A_code: T.int32 = arg_type_ids_1[0]
        assert var_A_code == 3 or var_A_code == 13 or var_A_code == 7 or var_A_code == 4, "matmul: Expect arg[0] to be pointer"
        var_B_code: T.int32 = arg_type_ids_1[1]
        assert var_B_code == 3 or var_B_code == 13 or var_B_code == 7 or var_B_code == 4, "matmul: Expect arg[1] to be pointer"
        var_matmul_code: T.int32 = arg_type_ids_1[2]
        assert var_matmul_code == 3 or var_matmul_code == 13 or var_matmul_code == 7 or var_matmul_code == 4, "matmul: Expect arg[2] to be pointer"
        var_A: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        var_B: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        var_matmul: T.handle = T.tvm_struct_get(args, 2, 12, "handle")
        assert not T.isnullptr(var_A), "matmul.var_A is expected to have non-NULL DLTensor* pointer"
        assert 2 == T.tvm_struct_get(var_A, 0, 4, "int32"), "matmul.var_A.ndim is expected to equal 2"
        matmul_var_A_shape: T.handle("int64") = T.tvm_struct_get(var_A, 0, 2, "handle")
        matmul_var_A_shape_1 = T.decl_buffer((2,), "int64", data=matmul_var_A_shape)
        matmul_var_A_strides: T.handle("int64") = T.tvm_struct_get(var_A, 0, 3, "handle")
        matmul_var_A_strides_1 = T.decl_buffer((0,), "int64", data=matmul_var_A_strides)
        dev_id: T.int32 = T.tvm_struct_get(var_A, 0, 9, "int32")
        A: T.handle("int32", "global") = T.tvm_struct_get(var_A, 0, 1, "handle")
        T.attr(A, "storage_alignment", 64)
        assert not T.isnullptr(var_B), "matmul.var_B is expected to have non-NULL DLTensor* pointer"
        assert 2 == T.tvm_struct_get(var_B, 0, 4, "int32"), "matmul.var_B.ndim is expected to equal 2"
        matmul_var_B_shape: T.handle("int64") = T.tvm_struct_get(var_B, 0, 2, "handle")
        matmul_var_B_shape_1 = T.decl_buffer((2,), "int64", data=matmul_var_B_shape)
        matmul_var_B_strides: T.handle("int64") = T.tvm_struct_get(var_B, 0, 3, "handle")
        matmul_var_B_strides_1 = T.decl_buffer((0,), "int64", data=matmul_var_B_strides)
        B: T.handle("int32", "global") = T.tvm_struct_get(var_B, 0, 1, "handle")
        T.attr(B, "storage_alignment", 64)
        assert not T.isnullptr(var_matmul), "matmul.var_matmul is expected to have non-NULL DLTensor* pointer"
        assert 2 == T.tvm_struct_get(var_matmul, 0, 4, "int32"), "matmul.var_matmul.ndim is expected to equal 2"
        matmul_var_matmul_shape: T.handle("int64") = T.tvm_struct_get(var_matmul, 0, 2, "handle")
        matmul_var_matmul_shape_1 = T.decl_buffer((2,), "int64", data=matmul_var_matmul_shape)
        matmul_var_matmul_strides: T.handle("int64") = T.tvm_struct_get(var_matmul, 0, 3, "handle")
        matmul_var_matmul_strides_1 = T.decl_buffer((0,), "int64", data=matmul_var_matmul_strides)
        matmul: T.handle("int32", "global") = T.tvm_struct_get(var_matmul, 0, 1, "handle")
        T.attr(matmul, "storage_alignment", 64)
        T.attr("default", "device_id", dev_id)
        T.attr("default", "device_type", 1)
        assert T.tvm_struct_get(var_A, 0, 5, "uint8") == T.uint8(0) and T.tvm_struct_get(var_A, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_A, 0, 7, "uint16") == T.uint16(1), "matmul.var_A.dtype is expected to be int32"
        assert matmul_var_A_shape_1[0] == T.int64(784), "Argument matmul.var_A.shape[0] has an unsatisfied constraint: T.int64(784) == matmul_var_A_shape[0]"
        assert matmul_var_A_shape_1[1] == T.int64(128), "Argument matmul.var_A.shape[1] has an unsatisfied constraint: T.int64(128) == matmul_var_A_shape[1]"
        if not T.isnullptr(matmul_var_A_strides):
            assert T.int64(1) == matmul_var_A_strides_1[1] and T.int64(128) == matmul_var_A_strides_1[0], "matmul.var_A.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_A, 0, 8, "uint64"), "Argument matmul.var_A.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_A, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_A, 0, 10, "int32") == 1, "Argument matmul.var_A.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_A, 0, 10, \"int32\")"
        assert not T.isnullptr(A), "matmul.var_A is expected to have non-NULL data pointer"
        assert T.tvm_struct_get(var_B, 0, 5, "uint8") == T.uint8(0) and T.tvm_struct_get(var_B, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_B, 0, 7, "uint16") == T.uint16(1), "matmul.var_B.dtype is expected to be int32"
        assert matmul_var_B_shape_1[0] == T.int64(128), "Argument matmul.var_B.shape[0] has an unsatisfied constraint: T.int64(128) == matmul_var_B_shape[0]"
        assert matmul_var_B_shape_1[1] == T.int64(10), "Argument matmul.var_B.shape[1] has an unsatisfied constraint: T.int64(10) == matmul_var_B_shape[1]"
        if not T.isnullptr(matmul_var_B_strides):
            assert T.int64(1) == matmul_var_B_strides_1[1] and T.int64(10) == matmul_var_B_strides_1[0], "matmul.var_B.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_B, 0, 8, "uint64"), "Argument matmul.var_B.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_B, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_B, 0, 10, "int32") == 1, "Argument matmul.var_B.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_B, 0, 10, \"int32\")"
        assert dev_id == T.tvm_struct_get(var_B, 0, 9, "int32"), "Argument matmul.var_B.device_id has an unsatisfied constraint: dev_id == T.tvm_struct_get(var_B, 0, 9, \"int32\")"
        assert not T.isnullptr(B), "matmul.var_B is expected to have non-NULL data pointer"
        assert T.tvm_struct_get(var_matmul, 0, 5, "uint8") == T.uint8(0) and T.tvm_struct_get(var_matmul, 0, 6, "uint8") == T.uint8(32) and T.tvm_struct_get(var_matmul, 0, 7, "uint16") == T.uint16(1), "matmul.var_matmul.dtype is expected to be int32"
        assert matmul_var_matmul_shape_1[0] == T.int64(784), "Argument matmul.var_matmul.shape[0] has an unsatisfied constraint: T.int64(784) == matmul_var_matmul_shape[0]"
        assert matmul_var_matmul_shape_1[1] == T.int64(10), "Argument matmul.var_matmul.shape[1] has an unsatisfied constraint: T.int64(10) == matmul_var_matmul_shape[1]"
        if not T.isnullptr(matmul_var_matmul_strides):
            assert T.int64(1) == matmul_var_matmul_strides_1[1] and T.int64(10) == matmul_var_matmul_strides_1[0], "matmul.var_matmul.strides: expected to be compact array"
            T.evaluate(0)
        assert T.uint64(0) == T.tvm_struct_get(var_matmul, 0, 8, "uint64"), "Argument matmul.var_matmul.byte_offset has an unsatisfied constraint: T.uint64(0) == T.tvm_struct_get(var_matmul, 0, 8, \"uint64\")"
        assert T.tvm_struct_get(var_matmul, 0, 10, "int32") == 1, "Argument matmul.var_matmul.device_type has an unsatisfied constraint: 1 == T.tvm_struct_get(var_matmul, 0, 10, \"int32\")"
        assert dev_id == T.tvm_struct_get(var_matmul, 0, 9, "int32"), "Argument matmul.var_matmul.device_id has an unsatisfied constraint: dev_id == T.tvm_struct_get(var_matmul, 0, 9, \"int32\")"
        assert not T.isnullptr(matmul), "matmul.var_matmul is expected to have non-NULL data pointer"
        A_1 = T.decl_buffer((T.int64(784), T.int64(128)), "int32", data=A)
        B_1 = T.decl_buffer((T.int64(128), T.int64(10)), "int32", data=B)
        matmul_1 = T.decl_buffer((T.int64(784), T.int64(10)), "int32", data=matmul)
        with T.attr(0, "compute_scope", "matmul_compute_"):
            for i0, i1, k in T.grid(784, 10, 128):
                cse_var_1: T.int32 = i0 * 10 + i1
                matmul_2 = T.Buffer((T.int64(7840),), "int32", data=matmul)
                if k == 0:
                    matmul_2[cse_var_1] = 0
                A_2 = T.Buffer((T.int64(100352),), "int32", data=A)
                B_2 = T.Buffer((T.int64(1280),), "int32", data=B)
                matmul_2[cse_var_1] = matmul_2[cse_var_1] + A_2[i0 * 128 + k] * B_2[k * 10 + i1]
        return 0

In [13]:
# NON-RELAX
my_instrument = MyInstrument()
with tvm.transform.PassContext(instruments=[my_instrument], config={"tir.usmp.enable": False}, opt_level=3):
    # ex = r.build(Module, target=tvm.target.Target("c", host="c"), runtime=tvm.relay.backend.Runtime("crt"), executor=tvm.relay.backend.Executor("aot", {"interface-api": "packed", "unpacked-api": False}))
    # ex = tvm.build(Module, target=tvm.target.Target("llvm", host="llvm"))
    ex = tvm.build(Module, target=tvm.target.Target("c", host="c"))

run_before_pass 0 sequential
['__class__', '__contains__', '__del__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_handle_by_constructor__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__setstate__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_add', '_import', '_move', '_relax_script', 'astext', 'attrs', 'clone', 'from_expr', 'functions', 'functions_items', 'get_attr', 'get_constructor', 'get_global_type_var', 'get_global_type_vars', 'get_global_var', 'get_global_vars', 'get_type', 'global_infos', 'global_type_var_map_', 'global_var_map_', 'handle', 'import_from_std', 'legacy_repr', 'same_as', 'script', 'show', 'source_map', 'type_definitions', 'update', 'update_func', 'update_global_info', 'w

[12:37:16] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[12:37:16] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[12:37:16] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[12:37:16] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListMoveFromPackedReturn
[12:37:16] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[12:37:16] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[12:37:16] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: 

In [14]:
print(ex.get_source())

// tvm target: c -keys=cpu 
#define TVM_EXPORTS
#include "tvm/runtime/c_runtime_api.h"
#include "tvm/runtime/c_backend_api.h"
#include <math.h>
#include <stdbool.h>
static void* vm_builtin_alloc_storage_packed = NULL;
static void* vm_builtin_alloc_tensor_packed = NULL;
static void* vm_builtin_null_value_packed = NULL;
static void* vm_builtin_copy_packed = NULL;
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t __vmtir__main(void* args, int32_t* arg_type_ids_1, int32_t num_args, void* out_ret_value, int32_t* out_ret_tcode, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t add(void* args, int32_t* arg_type_ids_1, int32_t num_args, void* out_ret_value, int32_t* out_ret_tcode, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t matmul(void* args, int32_t* arg_type_ids_1, int32_t num_args, void* out_ret_value, int32_t* out_ret_tcode, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t __vmtir__main(void* args, i

In [23]:
print(my_instrument.output[-9][0])

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    I.module_attrs({"runtime": None})
    @T.prim_func
    def add(args: T.handle, arg_type_ids_1: T.handle("int32", "global"), num_args: T.int32, out_ret_value: T.handle("void", "global"), out_ret_tcode: T.handle("int32", "global"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"host": {"keys": ["cpu"], "kind": "c", "tag": ""}, "keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""}), "tir.noalias": T.bool(True)})
        assert num_args == 3, "add: num_args should be 3"
        assert not T.isnullptr(args), "add: TVMValue* arg pointer was NULL"
        assert not T.isnullptr(arg_type_ids_1), "add: int* type_codes was NULL"
        arg_type_ids_1_1 = T.Buffer((3,), "int32", data=arg_type_ids_1)
        var_A_code: T.int32 = arg_type_ids_1_1[0]
        assert var_A_code == 3 or var_A_code == 13 or var_A_code == 7 

In [ ]:



    with tvm.transform.PassContext():
        ex = relax.build(mod, target=tvm.target.Target("c", host="c"), pipeline="micro2_build", exec_mode="crt", system_lib=True)



In [5]:
builder = relax.BlockBuilder()

# input_size = 784
# hidden_size = 128
# output_size = 10

input_n = 1
input_c = 16
input_h = 64
input_w = 64
kernel_h = 4
kernel_w = 4
kernel_ci = 16
kernel_co = 16
output_n = 1
output_c = 16
output_h = 61
output_w = 61

dtype = "float32"

weights_matrix = np.random.random((kernel_h, kernel_w, kernel_ci, kernel_co)).astype(dtype)
bias_matrix = np.random.random((output_w,)).astype(dtype)

with builder.function("main"):
    input = relax.Var("x", R.Tensor((input_n, input_c, input_h, input_w), dtype))
    weights = relax.Constant(tvm.nd.array(weights_matrix))
    bias = relax.Constant(tvm.nd.array(bias_matrix))
    output_conv2d = relax.op.nn.conv2d(input, weights, data_layout="NCHW", kernel_layout="HWIO")
    output_bias = relax.op.add(output_conv2d, bias)
    builder.emit_func_output(output_bias, params=[input])

mod2 = builder.get()
mod2.show()